In [ ]:
# 下载 Titanic 数据集：Kaggle 环境用现成路径，否则用 kaggle API 下载并解压
import os
from pathlib import Path

iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
if iskaggle: path = Path('../input/titanic')
else:
    path = Path('titanic')
    if not path.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(path))
        zipfile.ZipFile(f'{path}.zip').extractall(path)

In [ ]:
# 设置 numpy / torch / pandas 的打印宽度，方便查看数据
import torch, numpy as np, pandas as pd
np.set_printoptions(linewidth=140)
torch.set_printoptions(linewidth=140, sci_mode=False, edgeitems=7)
pd.set_option('display.width', 140)

In [ ]:
# 读取训练数据
df = pd.read_csv(path/'train.csv')
df

In [ ]:
# 统计每列缺失值数量
df.isna().sum()

In [ ]:
# 取每列的众数（第 0 行），用来填缺失
modes = df.mode().iloc[0]
modes

In [ ]:
# 用众数填充所有缺失值（原地修改）
df.fillna(modes, inplace=True)

In [ ]:
# 确认缺失值已填完
df.isna().sum()

In [ ]:
# 看数值列的统计信息
import numpy as np

df.describe(include=(np.number))

In [ ]:
# Fare 票价分布直方图（长尾严重）
df['Fare'].hist();

In [ ]:
# 对 Fare 取对数（+1 防止 log0），压缩长尾
df['LogFare'] = np.log(df['Fare']+1)

In [ ]:
# 取对数后分布更接近正态
df['LogFare'].hist();

In [ ]:
# 舱位等级 Pclass 的取值
pclasses = sorted(df.Pclass.unique())
pclasses

In [ ]:
# 看文本（object）列的统计
df.describe(include=[object])

In [ ]:
# 把类别列转成 one-hot 独热编码
df = pd.get_dummies(df, columns=["Sex","Pclass","Embarked"])
df.columns

In [ ]:
# 挑出新增的独热编码列看看
added_cols = ['Sex_male', 'Sex_female', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
df[added_cols].head()

In [ ]:
# 因变量（是否生还）转成张量
from torch import tensor

t_dep = tensor(df.Survived)

In [ ]:
# 自变量 = 连续列 + 独热列，转成 float 张量
indep_cols = ['Age', 'SibSp', 'Parch', 'LogFare'] + added_cols

t_indep = tensor(df[indep_cols].values, dtype=torch.float)
t_indep

In [ ]:
# 自变量张量的形状（行=样本数, 列=特征数）
t_indep.shape

In [ ]:
# 固定随机种子，随机初始化每个特征的系数（-0.5~0.5）
torch.manual_seed(442)

n_coeff = t_indep.shape[1]
coeffs = torch.rand(n_coeff)-0.5
coeffs

In [ ]:
# 特征 × 系数（逐元素相乘）
t_indep*coeffs

In [ ]:
# 按每列最大值归一化，避免大数值特征主导
vals,indices = t_indep.max(dim=0)
t_indep = t_indep / vals

In [ ]:
# 归一化后再看 特征 × 系数
t_indep*coeffs

In [ ]:
# 每行求和 = 每个样本的预测值
preds = (t_indep*coeffs).sum(axis=1)

In [ ]:
# 看前 10 个预测
preds[:10]

In [ ]:
# 损失 = 预测与真实的平均绝对误差（MAE）
loss = torch.abs(preds-t_dep).mean()
loss

In [ ]:
# 把"算预测"和"算损失"封装成函数
def calc_preds(coeffs, indeps): return (indeps*coeffs).sum(axis=1)
def calc_loss(coeffs, indeps, deps): return torch.abs(calc_preds(coeffs, indeps)-deps).mean()

In [ ]:
# 让系数开启梯度追踪（autograd）
coeffs.requires_grad_()

In [ ]:
# 算一次损失
loss = calc_loss(coeffs, t_indep, t_dep)
loss

In [ ]:
# 反向传播，求各系数的梯度
loss.backward()

In [ ]:
# 查看梯度
coeffs.grad

In [ ]:
# 再算一次损失并反传（注意梯度会累加）
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
coeffs.grad

In [ ]:
# 手动一步梯度下降：系数 -= 梯度×学习率，然后清零梯度
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
with torch.no_grad():
    coeffs.sub_(coeffs.grad * 0.1)
    coeffs.grad.zero_()
    print(calc_loss(coeffs, t_indep, t_dep))

In [ ]:
# 用 fastai 的 RandomSplitter 划分训练 / 验证集
from fastai.data.transforms import RandomSplitter
trn_split,val_split=RandomSplitter(seed=42)(df)

In [ ]:
# 按索引切出训练 / 验证的自变量与因变量
trn_indep,val_indep = t_indep[trn_split],t_indep[val_split]
trn_dep,val_dep = t_dep[trn_split],t_dep[val_split]
len(trn_indep),len(val_indep)

In [ ]:
# 更新系数的函数
def update_coeffs(coeffs, lr):
    coeffs.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()

In [ ]:
# 训练一个 epoch：算损失 → 反传 → 更新系数
def one_epoch(coeffs, lr):
    loss = calc_loss(coeffs, trn_indep, trn_dep)
    loss.backward()
    with torch.no_grad(): update_coeffs(coeffs, lr)
    print(f"{loss:.3f}", end="; ")

In [ ]:
# 初始化系数（并开启梯度）
def init_coeffs(): return (torch.rand(n_coeff)-0.5).requires_grad_()

In [ ]:
# 训练主循环（跑多个 epoch）
def train_model(epochs=30, lr=0.01):
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): one_epoch(coeffs, lr=lr)
    return coeffs

In [ ]:
# 训练 18 轮，学习率 0.2
coeffs = train_model(18, lr=0.2)

In [ ]:
# 查看各特征学到的系数
def show_coeffs(): return dict(zip(indep_cols, coeffs.requires_grad_(False)))
show_coeffs()

In [ ]:
# 用验证集做预测
preds = calc_preds(coeffs, val_indep)

In [ ]:
# 预测 >0.5 记为生还，和真实标签比较
results = val_dep.bool()==(preds>0.5)
results[:16]

In [ ]:
# 准确率 = 预测正确的比例
results.float().mean()

In [ ]:
# 把准确率算法封装成函数
def acc(coeffs): return (val_dep.bool()==(calc_preds(coeffs, val_indep)>0.5)).float().mean()
acc(coeffs)

In [ ]:
# 看前 28 个预测（注意有的超出 0~1，所以下面要用 sigmoid）
preds[:28]

In [ ]:
# 画 sigmoid 曲线：把任意实数压到 0~1
import sympy
sympy.plot("1/(1+exp(-x))", xlim=(-5,5));

In [ ]:
# 给预测套上 sigmoid，输出变成概率
def calc_preds(coeffs, indeps): return torch.sigmoid((indeps*coeffs).sum(axis=1))

In [ ]:
# 用更大的学习率重新训练
coeffs = train_model(lr=100)

In [ ]:
# 看准确率
acc(coeffs)

In [ ]:
# 看系数
show_coeffs()

In [ ]:
# 读测试集
tst_df = pd.read_csv(path/'test.csv')

In [ ]:
# 测试集 Fare 缺失先填 0
tst_df['Fare'] = tst_df.Fare.fillna(0)

In [ ]:
# 测试集做和训练集一样的预处理（填缺失 / 取对数 / 独热 / 归一化）
tst_df.fillna(modes, inplace=True)
tst_df['LogFare'] = np.log(tst_df['Fare']+1)
tst_df = pd.get_dummies(tst_df, columns=["Sex","Pclass","Embarked"])

tst_indep = tensor(tst_df[indep_cols].values, dtype=torch.float)
tst_indep = tst_indep / vals

In [ ]:
# 对测试集预测是否生还  # 修正：calc_preds 参数顺序是 (coeffs, indeps)，原来 (tst_indep, coeffs) 写反了会形状不匹配报错
tst_df['Survived'] = (calc_preds(coeffs, tst_indep)>0.5).int()

In [ ]:
# 生成 Kaggle 提交文件 sub.csv
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('sub.csv', index=False)

In [ ]:
# 看一下提交文件前几行
!head sub.csv

In [ ]:
# 逐元素相乘再求和
(val_indep*coeffs).sum(axis=1)

In [ ]:
# 直接用矩阵乘 @，结果相同但更简洁
val_indep@coeffs

In [ ]:
# 把 calc_preds 改写成矩阵乘版本
def calc_preds(coeffs, indeps): return torch.sigmoid(indeps@coeffs)

In [ ]:
# 系数改成列向量 (n_coeff, 1)
def init_coeffs(): return (torch.rand(n_coeff, 1)*0.1).requires_grad_()

In [ ]:
# 因变量也加一维，匹配矩阵乘后的输出形状
trn_dep = trn_dep[:,None]
val_dep = val_dep[:,None]

In [ ]:
# 重新训练
coeffs = train_model(lr=100)

In [ ]:
# 看准确率
acc(coeffs)

In [ ]:
# 初始化两层神经网络的参数
def init_coeffs(n_hidden=20):
    layer1 = (torch.rand(n_coeff, n_hidden)-0.5)/n_hidden
    layer2 = torch.rand(n_hidden, 1)-0.3
    const = torch.rand(1)[0]
    return layer1.requires_grad_(),layer2.requires_grad_(),const.requires_grad_()

In [ ]:
# 两层网络前向：ReLU 隐藏层 + sigmoid 输出
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    l1,l2,const = coeffs
    res = F.relu(indeps@l1)
    res = res@l2 + const
    return torch.sigmoid(res)

In [ ]:
# 逐层更新参数
def update_coeffs(coeffs, lr):
    for layer in coeffs:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [ ]:
# 训练（lr=1.4）
coeffs = train_model(lr=1.4)

In [ ]:
# 换更大的学习率再训练一次
coeffs = train_model(lr=20)

In [ ]:
# 初始化任意深度网络的参数（这里两个隐藏层，各 10 个神经元）
def init_coeffs():
    hiddens = [10, 10]  # <-- set this to the size of each hidden layer you want
    sizes = [n_coeff] + hiddens + [1]
    n = len(sizes)
    layers = [(torch.rand(sizes[i], sizes[i+1])-0.3)/sizes[i+1]*4 for i in range(n-1)]
    consts = [(torch.rand(1)[0]-0.5)*0.1 for i in range(n-1)]
    for l in layers+consts: l.requires_grad_()
    return layers,consts

In [ ]:
# 通用的多层前向传播
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    layers,consts = coeffs
    n = len(layers)
    res = indeps
    for i,l in enumerate(layers):
        res = res@l + consts[i]
        if i!=n-1: res = F.relu(res)
    return torch.sigmoid(res)

In [ ]:
# 更新所有层的参数
def update_coeffs(coeffs, lr):
    layers,consts = coeffs
    for layer in layers+consts:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [ ]:
# 训练这个更深的网络
coeffs = train_model(lr=4)

In [ ]:
# 看最终准确率
acc(coeffs)